## Financial Transaction

#### Bloque 1: Carga y validación de datos

##### Paso 1 — Importar librerías y cargar las 3 tablas
Antes de analizar nada, necesitamos cargar los tres CSV en memoria como DataFrames de pandas. 
Uso pandas porque es el estándar para manipulación de datos tabulares en Python, y de paso importo numpy (por si lo necesitamos más adelante para cálculos) 
 y OS para gestionar rutas de archivo de forma segura

In [1]:
import pandas as pd
import numpy as np
import os

# Ruta donde tengas los 3 CSV descargados de Kaggle (ajusta esta línea a tu carpeta local)
ruta_datos = "./data/"

users = pd.read_csv(os.path.join(ruta_datos, "users_data.csv"))
cards = pd.read_csv(os.path.join(ruta_datos, "cards_data.csv"))
transactions = pd.read_csv(os.path.join(ruta_datos, "transactions_data.csv"))

print("Usuarios:", users.shape)
print("Tarjetas:", cards.shape)
print("Transacciones:", transactions.shape)

Usuarios: (2000, 14)
Tarjetas: (6146, 13)
Transacciones: (13305915, 12)


##### Paso 2 - Inspección inicial de cada tabla
Antes de tocar nada, necesitamos ver qué tenemos: nombres de columnas, tipos de datos y una muestra de filas. Esto es equivalente a lo primero que hacíamos al abrir las tablas en Power Query — entender la estructura antes de transformar.

In [2]:
for nombre, df in [("users", users), ("cards", cards), ("transactions", transactions)]:
    print(f"\n===== {nombre.upper()} =====")
    print(df.info())
    print(df.head(3))


===== USERS =====
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   str    
 6   address            2000 non-null   str    
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   str    
 10  yearly_income      2000 non-null   str    
 11  total_debt         2000 non-null   str    
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 218.9 KB
None
     id  current_age  retirement_age  birth_year  bi

In [3]:
# Limpiar columnas monetarias: quitar el símbolo $ y convertir a float
columnas_dinero = ["per_capita_income", "yearly_income", "total_debt"]

for col in columnas_dinero:
    users[col] = users[col].replace('[\$,]', '', regex=True).astype(float)

# Verificamos que el cambio funcionó
print(users[columnas_dinero].dtypes)
print(users[columnas_dinero].head(3))

per_capita_income    float64
yearly_income        float64
total_debt           float64
dtype: object
   per_capita_income  yearly_income  total_debt
0            29278.0        59696.0    127613.0
1            37891.0        77254.0    191349.0
2            22681.0        33483.0       196.0


<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
C:\Users\JAVI SM\AppData\Local\Temp\ipykernel_3648\1890142973.py:5: SyntaxWarning: invalid escape sequence '\$'
  users[col] = users[col].replace('[\$,]', '', regex=True).astype(float)


##### Funcionó — las tres columnas ya son float64 y los valores se ven correctos (29278.0, 59696.0, etc.). El único detalle es el SyntaxWarning en amarillo/rojo: no es un error que rompa el código, pero conviene arreglarlo para dejarlo limpio.
Por qué pasa: en el patrón '[\$,]', la barra invertida \ antes del $ es una secuencia de escape que Python no reconoce como válida en un string normal (aunque sí funciona por casualidad en expresiones regulares). La forma correcta es usar un raw string (con una r delante), que le dice a Python "trata este texto literal, no interpretes las barras invertidas".

In [4]:
columnas_dinero = ["per_capita_income", "yearly_income", "total_debt"]

for col in columnas_dinero:
    users[col] = users[col].replace(r'[\$,]', '', regex=True).astype(float)

print(users[columnas_dinero].dtypes)

per_capita_income    float64
yearly_income        float64
total_debt           float64
dtype: object


##### Inspeccionamos la tabla cards

In [5]:
print(cards.info())
print(cards.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 6146 entries, 0 to 6145
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id                     6146 non-null   int64
 1   client_id              6146 non-null   int64
 2   card_brand             6146 non-null   str  
 3   card_type              6146 non-null   str  
 4   card_number            6146 non-null   int64
 5   expires                6146 non-null   str  
 6   cvv                    6146 non-null   int64
 7   has_chip               6146 non-null   str  
 8   num_cards_issued       6146 non-null   int64
 9   credit_limit           6146 non-null   str  
 10  acct_open_date         6146 non-null   str  
 11  year_pin_last_changed  6146 non-null   int64
 12  card_on_dark_web       6146 non-null   str  
dtypes: int64(6), str(7)
memory usage: 624.3 KB
None
     id  client_id card_brand card_type       card_number  expires  cvv  \
0  4524        825       V

In [6]:
# Limpiar la columna de límite de crédito, igual que hicimos con las de ingresos
cards["credit_limit"] = cards["credit_limit"].replace(r'[\$,]', '', regex=True).astype(float)

print(cards["credit_limit"].dtype)
print(cards["credit_limit"].head(3))

float64
0    24295.0
1    21968.0
2    46414.0
Name: credit_limit, dtype: float64


In [7]:
# Convertir las columnas de fecha, indicando el formato exacto MM/YYYY
cards["expires"] = pd.to_datetime(cards["expires"], format="%m/%Y")
cards["acct_open_date"] = pd.to_datetime(cards["acct_open_date"], format="%m/%Y")

print(cards[["expires", "acct_open_date"]].dtypes)
print(cards[["expires", "acct_open_date"]].head(3))

expires           datetime64[us]
acct_open_date    datetime64[us]
dtype: object
     expires acct_open_date
0 2022-12-01     2002-09-01
1 2020-12-01     2014-04-01
2 2024-02-01     2003-07-01


In [8]:
# Convertir columnas YES/NO a booleano
cards["has_chip"] = cards["has_chip"].map({"YES": True, "NO": False})
cards["card_on_dark_web"] = cards["card_on_dark_web"].map({"YES": True, "NO": False})

print(cards[["has_chip", "card_on_dark_web"]].dtypes)
print(cards[["has_chip", "card_on_dark_web"]].head(3))

has_chip              bool
card_on_dark_web    object
dtype: object
   has_chip card_on_dark_web
0      True              NaN
1      True              NaN
2      True              NaN


##### Aquí hay un problema: has_chip se convirtió bien a bool, pero card_on_dark_web salió como object con NaN en las tres primeras filas — el .map() no encontró coincidencia con "NO".
Esto normalmente pasa porque los valores reales no son exactamente "YES"/"NO" como escribimos (puede haber espacios en blanco, minúsculas, o algún valor distinto). Antes de asumir nada, vamos a comprobar qué valores únicos tiene realmente esa columna:

In [9]:
# Revisamos los valores únicos reales de la columna antes de mapear
print(cards["card_on_dark_web"].unique())

[nan]


##### Ahí está la explicación — y es un fallo mío que se me pasó advertirte: cuando ejecutaste el .map() anterior, ya sobrescribiste la columna original con el resultado (todo NaN), así que ahora .unique() solo puede ver lo que quedó, que es NaN para todos. Ya no tenemos forma de recuperar los valores originales desde cards en memoria — hay que recargarlos.
Solución: releer la columna original desde el CSV (sin tener que recargar toda la tabla de 13M de filas, esto es solo cards, así que es rápido):

In [10]:
# Recargamos solo la columna original desde el CSV para ver sus valores reales
card_dark_web_original = pd.read_csv(os.path.join(ruta_datos, "cards_data.csv"), usecols=["card_on_dark_web"])
print(card_dark_web_original["card_on_dark_web"].unique())

<StringArray>
['No']
Length: 1, dtype: str


##### Ahí está el problema: el valor real es 'No' (con mayúscula solo en la N), no 'NO' en mayúsculas como escribimos en el .map() — por eso no encontró coincidencia y todo quedó en NaN.
Además, un dato interesante: Length: 1 significa que en toda la columna solo existe el valor 'No' — ningún registro tiene 'Yes'. Es decir, ninguna tarjeta de este dataset está marcada como comprometida en la dark web. Es útil saberlo: esta columna, tal cual está, no aporta variabilidad para el análisis (es una constante), aunque no cuesta nada dejarla bien formateada por si el dataset se actualiza o filtras de otra forma.
Para evitar este tipo de error de mayúsculas/minúsculas en el futuro, lo más robusto es normalizar el texto (quitar espacios y pasar a minúsculas) antes de mapear, así no importa si viene como "No", "NO" o "no":

In [11]:
# Recargamos la columna completa correctamente y la volvemos a añadir a cards
cards["card_on_dark_web"] = pd.read_csv(
    os.path.join(ruta_datos, "cards_data.csv"), usecols=["card_on_dark_web"]
)["card_on_dark_web"]

# Normalizamos texto (minúsculas, sin espacios) antes de mapear, para evitar el mismo error
cards["card_on_dark_web"] = cards["card_on_dark_web"].str.strip().str.lower().map({"yes": True, "no": False})

print(cards["card_on_dark_web"].unique())
print(cards["card_on_dark_web"].dtype)

[False]
bool


##### Inspeccionamos la tabla transactions

In [12]:
print(transactions.info())

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            str    
 2   client_id       int64  
 3   card_id         int64  
 4   amount          str    
 5   use_chip        str    
 6   merchant_id     int64  
 7   merchant_city   str    
 8   merchant_state  str    
 9   zip             float64
 10  mcc             int64  
 11  errors          str    
dtypes: float64(1), int64(5), str(6)
memory usage: 1.2 GB
None


##### Aquí notamos algo importante: la tabla no muestra la columna "Non-Null Count" como en las tablas anteriores — eso es porque pandas, al ser una tabla tan grande (13,3M filas), omite ese cálculo por defecto para no ralentizar la ejecución. Así que no sabemos aún si hay nulos, hay que comprobarlo aparte con un método más directo.
Lo que sí vemos de estructura:

client_id y card_id → serán las claves para unir con users y cards respectivamente. Buena señal, confirma la cadena de joins que planeamos (users → cards → transactions).
amount → está como str, casi seguro con el símbolo $ otra vez (misma limpieza que ya hemos hecho dos veces).
date → como str, habrá que convertir a datetime.
errors → como str, sospecho que aquí sí puede haber muchos nulos genuinos (no todas las transacciones tienen error), a diferencia de los casos anteriores.

Vamos a comprobar nulos y valores de forma controlada, evitando operaciones lentas innecesarias sobre las 13M filas:

In [13]:
# Contar nulos por columna (puede tardar unos segundos por el volumen)
print(transactions.isnull().sum())

id                       0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state     1563700
zip                1652706
mcc                      0
errors            13094522
dtype: int64


##### Ahora sí tenemos el panorama completo. Tres columnas con nulos, y cada una con una explicación de negocio distinta — importante no tratarlas todas igual:
errors: 13.094.522 nulos de 13.305.915 filas (98,4%). Esto no es un problema de calidad de datos, es información legítima: la inmensa mayoría de transacciones simplemente no tuvo ningún error, así que el campo queda vacío. Es coherente con lo que vimos en el resumen ejecutivo del proyecto anterior (la normalidad es la mayoría, el error es la excepción). No hay que "arreglar" esto, solo interpretarlo bien — un nulo aquí significa "sin incidencia", no un dato perdido.

merchant_state: 1.563.700 nulos (~11,7%). Probablemente corresponde a compras online o transacciones internacionales donde no aplica un estado/provincia de EE.UU. — también información legítima, no un fallo.

zip: 1.652.706 nulos (~12,4%). Mismo motivo probable que merchant_state — comercios sin código postal aplicable (online, extranjero).
Antes de decidir qué hacer con cada una, vale la pena confirmar la hipótesis de que merchant_state y zip están vacíos en los mismos casos (comercios online/extranjeros) y no por separado. Vamos a comprobarlo cruzando con use_chip, que probablemente indica si la transacción fue online, con chip o con banda:

In [14]:
# Revisamos los valores únicos de use_chip, y si coinciden con los nulos de merchant_state
print(transactions["use_chip"].unique())
print(transactions[transactions["merchant_state"].isnull()]["use_chip"].value_counts())

<StringArray>
['Swipe Transaction', 'Online Transaction', 'Chip Transaction']
Length: 3, dtype: str
use_chip
Online Transaction    1557912
Chip Transaction         5788
Name: count, dtype: int64


##### Confirma la hipótesis casi por completo: de los 1.563.700 nulos en merchant_state, 1.557.912 son transacciones online (Online Transaction) — tiene toda la lógica, una compra online no tiene un estado físico asociado al comercio.

Pero hay un matiz: quedan 5.788 casos que son Chip Transaction (compra física, con tarjeta presente) y aun así no tienen estado registrado. Esto ya no se explica solo por "es online", así que hay dos hipótesis a considerar:

Son transacciones físicas fuera de EE.UU. (donde no aplica el concepto de "state").
Son datos incompletos genuinos del comercio.

No es un problema grave (son solo el 0,04% de las 13,3M filas), pero antes de decidir cómo tratarlos, veamos si tienen algo en común, por ejemplo revisando zip en esos mismos casos:

In [15]:
# Revisamos los 5.788 casos de Chip Transaction sin merchant_state
casos_raros = transactions[(transactions["merchant_state"].isnull()) & (transactions["use_chip"] == "Chip Transaction")]
print(casos_raros[["merchant_city", "merchant_state", "zip"]].head(10))

        merchant_city merchant_state  zip
6573529        ONLINE            NaN  NaN
6573565        ONLINE            NaN  NaN
6573594        ONLINE            NaN  NaN
6573677        ONLINE            NaN  NaN
6574120        ONLINE            NaN  NaN
6574552        ONLINE            NaN  NaN
6574623        ONLINE            NaN  NaN
6574710        ONLINE            NaN  NaN
6575008        ONLINE            NaN  NaN
6575584        ONLINE            NaN  NaN


##### Interesante — esto revela algo que no esperábamos: aunque filtramos por use_chip == "Chip Transaction", la columna merchant_city dice "ONLINE" en estos casos. Es decir, hay una inconsistencia entre columnas: el campo use_chip dice que fue una transacción con chip (presencial), pero merchant_city dice que fue online. Ambas cosas no pueden ser ciertas a la vez.

Esto ya no es un caso de "nulo con explicación de negocio" como los anteriores — es un problema real de calidad de datos que vale la pena documentar (no necesariamente arreglar, pero sí dejar constancia). Puede deberse a un error de captura en el sistema origen del banco, algo bastante común en datasets reales.

Antes de decidir qué hacer, confirmemos el alcance exacto: ¿estos 5.788 casos son todos así (merchant_city = "ONLINE" pese a decir Chip Transaction), o solo una parte?

In [16]:
# Confirmamos si TODOS los casos raros tienen merchant_city = "ONLINE"
print(casos_raros["merchant_city"].value_counts())

merchant_city
ONLINE    5788
Name: count, dtype: int64


##### Confirmado al 100%: los 5.788 casos son todos merchant_city = "ONLINE". Con esto cerramos la investigación de merchant_state y zip:

Conclusión: los nulos en merchant_state y zip (1.563.700 y 1.652.706 respectivamente) se explican por completo por transacciones online, donde no aplica un estado/código postal físico. Hay una inconsistencia puntual en use_chip para 5.788 de esos casos (etiquetados como "Chip Transaction" pese a ser online), pero no afecta a la explicación del nulo — solo queda documentada como una peculiaridad del dato de origen, sin necesidad de corregirla porque no tenemos forma fiable de saber cuál campo está equivocado.

Decisión sobre estas columnas: no vamos a rellenar los nulos con ningún valor inventado — los dejamos como NaN, porque representan ausencia de información legítima (no aplica), no un dato perdido que haya que imputar. Igual con errors.

##### Con esto termina la revisión de nulos. Ahora nos falta lo mismo que hicimos en cards: limpiar tipos de datos. 
Empezamos por amount (el símbolo $, ya conocido) y date (a datetime):

In [17]:
# Limpiar amount (quitar $ y convertir a float)
transactions["amount"] = transactions["amount"].replace(r'[\$,]', '', regex=True).astype(float)

# Convertir date a datetime
transactions["date"] = pd.to_datetime(transactions["date"])

print(transactions[["amount", "date"]].dtypes)
print(transactions[["amount", "date"]].head(3))

amount           float64
date      datetime64[us]
dtype: object
   amount                date
0  -77.00 2010-01-01 00:01:00
1   14.57 2010-01-01 00:02:00
2   80.00 2010-01-01 00:02:00


##### Ambas columnas convertidas correctamente: amount como float64 y date como datetime64.

Un detalle a señalar antes de seguir: en el primer registro ves amount = -77.00. Esto no es un error, son importes negativos legítimos — probablemente representan devoluciones, reembolsos o correcciones (algo habitual en datasets de transacciones). Es bueno tenerlo en cuenta para más adelante: cuando calculemos "gasto total por usuario" en el bloque 2, tendremos que decidir si sumamos estos negativos tal cual (reflejando el neto real) o los tratamos aparte — pero eso lo abordamos cuando lleguemos a ese análisis, no ahora.

Con esto termina la limpieza de tipos de datos en las 3 tablas. Solo nos queda el último paso del Bloque 1: verificar la integridad de las claves antes de hacer los joins — es decir, comprobar que no haya client_id en cards que no exista en users, ni client_id/card_id en transactions que no existan en las tablas padre (el mismo tipo de chequeo de "IDs huérfanos" que hicimos en el proyecto de riesgo crediticio).

In [18]:
# ¿Hay client_id en cards que no existan en users?
ids_huerfanos_cards = cards[~cards["client_id"].isin(users["id"])]
print("Tarjetas con client_id huérfano:", len(ids_huerfanos_cards))

# ¿Hay client_id en transactions que no existan en users?
ids_huerfanos_trans_client = transactions[~transactions["client_id"].isin(users["id"])]
print("Transacciones con client_id huérfano:", len(ids_huerfanos_trans_client))

# ¿Hay card_id en transactions que no existan en cards?
ids_huerfanos_trans_card = transactions[~transactions["card_id"].isin(cards["id"])]
print("Transacciones con card_id huérfano:", len(ids_huerfanos_trans_card))

Tarjetas con client_id huérfano: 0
Transacciones con client_id huérfano: 0
Transacciones con card_id huérfano: 0


##### Excelente resultado — cero IDs huérfanos en las tres comprobaciones. Esto confirma que la integridad referencial entre las tablas es perfecta:

Todas las tarjetas pertenecen a un usuario existente.
Todas las transacciones pertenecen a un cliente existente.
Todas las transacciones pertenecen a una tarjeta existente.

Con esto, el Bloque 1 queda oficialmente cerrado. Resumen de lo que hemos hecho:
✅ Carga de las 3 tablas (users, cards, transactions)
✅ Inspección de estructura y tipos de datos
✅ Limpieza de columnas monetarias ($ → float) en las 3 tablas
✅ Conversión de fechas a datetime
✅ Conversión de columnas booleanas (has_chip, card_on_dark_web)
✅ Investigación y explicación de nulos (errors, merchant_state, zip) — todos con justificación de negocio, sin necesidad de imputar
✅ Verificación de integridad de claves entre las 3 tablas — sin huérfanos

#### Bloque 2 - Análisis de comportamioento de gasto (nivel usuario)

##### Paso 1 — Decidir cómo tratar los importes negativos

Antes de agregar nada, hay que resolver lo que dejamos pendiente: los importes negativos en amount (devoluciones/reembolsos). Si los sumamos tal cual junto con los positivos, obtenemos el gasto neto (lo que realmente salió de la cuenta del cliente). Si los excluimos, obtenemos el gasto bruto (solo lo gastado, ignorando devoluciones).

Para un análisis de comportamiento de gasto, lo más correcto es quedarnos con el neto como métrica principal, pero calculando también el número de devoluciones aparte, porque es información de comportamiento en sí misma (un cliente con muchas devoluciones es un patrón distinto a uno sin ninguna).

In [19]:
# Verificamos cuántas transacciones son negativas, para dimensionar el patrón
num_negativas = (transactions["amount"] < 0).sum()
pct_negativas = num_negativas / len(transactions) * 100
print(f"Transacciones negativas: {num_negativas} ({pct_negativas:.2f}%)")

Transacciones negativas: 660049 (4.96%)


##### Un 4,96% — no es marginal, pero tampoco domina el patrón. 
Confirma que vale la pena tratarlo como comentamos: calcular el gasto neto como métrica principal, y el número/importe de devoluciones como métrica aparte para no perder esa información de comportamiento.

##### Paso 2 — Unir las 3 tablas (el join encadenado)
Ahora sí, hacemos el join que estábamos preparando: transactions → cards → users. 
Empezamos por transactions, porque es la tabla base sobre la que queremos añadir el contexto de tarjeta y usuario.

Usamos merge() de pandas, que es el equivalente a un JOIN de SQL. Como confirmamos que no hay IDs huérfanos, un inner join (por defecto en merge) no perderá ninguna fila — pero lo indicamos explícito con how="inner" para dejar claro que es intencional, no un descuido.

In [20]:
# Paso 1: unir transactions con cards (por card_id = id de la tarjeta)
trans_cards = transactions.merge(
    cards[["id", "client_id", "card_brand", "card_type", "credit_limit"]],
    left_on="card_id", right_on="id",
    how="inner",
    suffixes=("", "_card")
)

# Paso 2: unir el resultado con users (por client_id)
trans_completo = trans_cards.merge(
    users[["id", "current_age", "gender", "yearly_income", "total_debt", "credit_score"]],
    left_on="client_id", right_on="id",
    how="inner",
    suffixes=("", "_user")
)

print(trans_completo.shape)
print(trans_completo.columns.tolist())

(13305915, 23)
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'id_card', 'client_id_card', 'card_brand', 'card_type', 'credit_limit', 'id_user', 'current_age', 'gender', 'yearly_income', 'total_debt', 'credit_score']


##### El join salió perfecto: 13.305.915 filas — exactamente las mismas que teníamos en transactions original, confirmando que no se perdió ni se duplicó ninguna fila en el proceso (como esperábamos, dado que ya habíamos verificado cero IDs huérfanos).

Veo un pequeño detalle en los nombres de columnas por los sufijos automáticos: ahora tenemos client_id (el original de transactions) y también id_card, client_id_card, id_user — columnas redundantes que ya cumplieron su función de unión pero que no necesitamos conservar.
Vale la pena limpiarlas para que el DataFrame quede más manejable:

In [21]:
# Eliminamos columnas redundantes que quedaron del proceso de merge
trans_completo = trans_completo.drop(columns=["id_card", "client_id_card", "id_user"])

print(trans_completo.shape)
print(trans_completo.columns.tolist())

(13305915, 20)
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_brand', 'card_type', 'credit_limit', 'current_age', 'gender', 'yearly_income', 'total_debt', 'credit_score']


##### Paso 3 — Agregación: gasto por usuario

Ahora sí, construimos la primera tabla resumen del bloque: gasto neto total, número de transacciones, ticket medio, y el detalle de devoluciones, todo agrupado por client_id.

Usamos .groupby() con .agg(), que nos permite calcular varias métricas distintas en un solo paso, cada una sobre la columna que corresponda:

In [22]:
resumen_gasto_usuario = trans_completo.groupby("client_id").agg(
    gasto_neto_total=("amount", "sum"),
    num_transacciones=("amount", "count"),
    ticket_medio=("amount", "mean"),
    num_devoluciones=("amount", lambda x: (x < 0).sum()),
    importe_devoluciones=("amount", lambda x: x[x < 0].sum())
).reset_index()

print(resumen_gasto_usuario.shape)
print(resumen_gasto_usuario.head(5))

(1219, 6)
   client_id  gasto_neto_total  num_transacciones  ticket_medio  \
0          0         625799.67              12795     48.909705   
1          1         336187.37              10073     33.375099   
2          2         291534.27              10612     27.472132   
3          3         280685.46               6001     46.773114   
4          4         595722.36              15043     39.601300   

   num_devoluciones  importe_devoluciones  
0               635              -77560.0  
1               142              -15867.0  
2               508              -40413.0  
3                54               -6125.0  
4               657              -67565.0  


##### Aquí .reset_index() convierte client_id (que groupby deja como índice) de nuevo en una columna normal — más cómodo para trabajar y para exportar después a Excel.

##### Los cálculos se ven coherentes (gasto neto, ticket medio, devoluciones todo con sentido), pero hay algo que hay que parar a revisar antes de seguir: el resultado tiene 1.219 filas, cuando users tiene 2.000 usuarios. Eso significa que hay 781 usuarios que no aparecen en ninguna transacción.

No es necesariamente un error — simplemente son clientes que tienen tarjeta pero no han hecho ninguna transacción registrada en este dataset — pero es importante confirmarlo antes de seguir, porque afecta a cómo interpretamos y presentamos los datos más adelante (por ejemplo, si luego unimos esta tabla con users usando inner join, esos 781 clientes desaparecerían silenciosamente del análisis).

Vamos a confirmarlo:

In [23]:
# Confirmamos cuántos usuarios no tienen ninguna transacción
usuarios_sin_transacciones = users[~users["id"].isin(trans_completo["client_id"])]
print("Usuarios sin transacciones:", len(usuarios_sin_transacciones))
print(usuarios_sin_transacciones[["id", "current_age", "yearly_income"]].head(5))

Usuarios sin transacciones: 781
      id  current_age  yearly_income
7   1711           26        54623.0
10   192           27        56164.0
11   640           29        45727.0
12  1679           18        69149.0
14  1590           48        20513.0


##### Confirmado: 781 usuarios sin ninguna transacción (39% de la base de clientes) — un porcentaje demasiado alto para ignorarlo sin más. Viendo las edades (26, 27, 29, 18, 48), no parece un patrón por edad concreta, así que probablemente sean simplemente clientes con tarjeta emitida pero sin actividad en el periodo del dataset.

Vamos a corregirlo como comentamos: usamos un left join desde users hacia el resumen de gasto, para que estos 781 clientes aparezcan con valores en 0 en vez de desaparecer del análisis.

In [24]:
# Left join: partimos de TODOS los usuarios, no solo los que tienen transacciones
resumen_gasto_usuario_completo = users[["id", "current_age", "gender", "yearly_income", "total_debt", "credit_score"]].merge(
    resumen_gasto_usuario,
    left_on="id", right_on="client_id",
    how="left"
)

# Rellenamos con 0 las métricas de gasto para quienes no tienen transacciones
columnas_a_rellenar = ["gasto_neto_total", "num_transacciones", "ticket_medio", "num_devoluciones", "importe_devoluciones"]
resumen_gasto_usuario_completo[columnas_a_rellenar] = resumen_gasto_usuario_completo[columnas_a_rellenar].fillna(0)

# Quitamos la columna client_id duplicada (ya tenemos "id")
resumen_gasto_usuario_completo = resumen_gasto_usuario_completo.drop(columns=["client_id"])

print(resumen_gasto_usuario_completo.shape)
print(resumen_gasto_usuario_completo["num_transacciones"].eq(0).sum(), "usuarios con 0 transacciones")
resumen_gasto_usuario_completo.head(5)

(2000, 11)
781 usuarios con 0 transacciones


,id,current_age,gender,yearly_income,total_debt,credit_score,gasto_neto_total,num_transacciones,ticket_medio,num_devoluciones,importe_devoluciones
0,825,53,Female,59696.0,127613.0,787,912398.30,11376.0,80.203789,369.0,-53012.0
1,1746,53,Female,77254.0,191349.0,701,446774.24,5593.0,79.880965,261.0,-46712.0
2,1718,81,Female,33483.0,196.0,698,856419.68,25890.0,33.079169,371.0,-48776.0
3,708,63,Female,249925.0,202328.0,722,1094355.64,8681.0,126.063315,464.0,-87193.0
4,1164,43,Male,109687.0,183855.0,675,890509.67,9227.0,96.511290,138.0,-34178.0


##### Perfecto, ambas confirmaciones correctas: 2.000 filas (todos los usuarios) y 781 con 0 transacciones, exactamente lo esperado. La tabla resumen_gasto_usuario_completo ya está lista y es honesta con toda la base de clientes.

Con esto tenemos completado el primer gran resultado del Bloque 2: comportamiento de gasto por usuario, cruzado con su perfil demográfico (edad, género, ingresos, deuda, credit score).

##### Siguiente paso — Categorías de comercio más frecuentes por usuario

Según lo que planeamos, el siguiente punto del bloque 2 es ver qué categorías de comercio (mcc) predominan en el gasto de cada usuario. Para interpretar el código mcc (que ahora mismo es solo un número) necesitaríamos el archivo mcc_codes.json que mencionamos al principio, para traducirlo a una categoría legible (ej. "Restaurantes", "Supermercados").

##### Cargar y explorar el archivo mcc_codes.json
Antes de mapear nada, necesitamos ver la estructura del JSON: normalmente es un diccionario donde la clave es el código MCC (número) y el valor es la descripción de la categoría (texto). Lo cargamos con la librería json de Python (no hace falta pandas para esto, es más directo así).

In [25]:
import json

with open(os.path.join(ruta_datos, "mcc_codes.json"), "r") as f:
    mcc_codes = json.load(f)

# Vemos cómo está estructurado (primeros 5 elementos)
print(list(mcc_codes.items())[:5])
print("Total de categorías:", len(mcc_codes))

[('5812', 'Eating Places and Restaurants'), ('5541', 'Service Stations'), ('7996', 'Amusement Parks, Carnivals, Circuses'), ('5411', 'Grocery Stores, Supermarkets'), ('4784', 'Tolls and Bridge Fees')]
Total de categorías: 109


##### Confirmado el formato: las claves son strings (ej. '5812', no el número 5812), con 109 categorías en total. Esto es justo el tipo de detalle que hay que verificar antes de mapear, porque la columna mcc en transactions es int64 (número), así que hay un desajuste de tipos que tenemos que resolver antes de que el .map() funcione.

##### Paso 2 — Mapear el código MCC a su categoría
Convertimos la columna mcc a string antes de mapear (o convertimos las claves del diccionario a int — cualquiera de las dos funciona, pero es más seguro convertir la columna, ya que sabemos con certeza que el JSON usa strings):

In [26]:
# Convertimos mcc a string para que coincida con las claves del diccionario
trans_completo["mcc_categoria"] = trans_completo["mcc"].astype(str).map(mcc_codes)

# Comprobamos si quedó algún código sin traducir (mapeo fallido)
print("Categorías sin traducir (NaN):", trans_completo["mcc_categoria"].isnull().sum())
print(trans_completo[["mcc", "mcc_categoria"]].head(5))

Categorías sin traducir (NaN): 0
    mcc                          mcc_categoria
0  5499              Miscellaneous Food Stores
1  5311                      Department Stores
2  4829                         Money Transfer
3  4829                         Money Transfer
4  5813  Drinking Places (Alcoholic Beverages)


##### Paso 3 — Construir la tabla amplia: gasto por usuario y categoría
Ahora sí, la tabla que querías para cruzar en Excel: gasto e importe por cada combinación de usuario × categoría de comercio. Usamos .groupby() con dos claves (client_id y mcc_categoria) en vez de una:

In [27]:
gasto_usuario_categoria = trans_completo.groupby(["client_id", "mcc_categoria"]).agg(
    gasto_total=("amount", "sum"),
    num_transacciones=("amount", "count"),
    ticket_medio=("amount", "mean")
).reset_index()

print(gasto_usuario_categoria.shape)
print(gasto_usuario_categoria.head(10))

(93238, 5)
   client_id                                      mcc_categoria  gasto_total  \
0          0                                           Airlines      2862.64   
1          0               Amusement Parks, Carnivals, Circuses      2593.56   
2          0                  Artist Supply Stores, Craft Shops      5403.35   
3          0                 Athletic Fields, Commercial Sports        87.08   
4          0                           Automotive Service Shops     37048.23   
5          0                            Beauty and Barber Shops      6939.53   
6          0                                        Book Stores      6715.87   
7          0                Brick, Stone, and Related Materials      2073.99   
8          0  Cable, Satellite, and Other Pay Television Ser...      5861.30   
9          0                                         Car Washes      1249.80   

   num_transacciones  ticket_medio  
0                  6    477.106667  
1                 12    216.130000

##### Con esto obtienes una fila por cada combinación cliente-categoría que realmente exista (no todas las combinaciones posibles, solo las que tienen al menos una transacción) — es el formato ideal para una tabla dinámica en Excel: pones client_id en filas, mcc_categoria en columnas, y gasto_total como valores, y tendrás el cruce completo interactivo.

##### Para la segmentación de ingresos, recomiendo hacerlo en Python con cuartiles (bajo/medio-bajo/medio-alto/alto), en vez de dejarlo continuo.
La razón es que definir los cortes de segmentación es una decisión analítica (dónde trazamos la línea entre "ingresos bajos" y "medios"), y es mejor que quede documentada y reproducible en el código, en vez de que cada persona que abra el Excel decida sus propios cortes de forma distinta cada vez. Además, así entregamos una columna ya lista para usar como filtro/segmentador en la tabla dinámica, sin que el usuario de Excel tenga que calcular nada.

##### Paso 1 - Crear los segmentos de ingresos por cuartiles

##### pd.qcut() divide automáticamente en 4 grupos de tamaño similar según los valores reales de la columna (no rangos fijos inventados a mano), así que el resultado se adapta a la distribución real de ingresos de este dataset.

In [28]:
# Creamos 4 segmentos de ingresos basados en cuartiles (25% de usuarios en cada grupo)
resumen_gasto_usuario_completo["segmento_ingresos"] = pd.qcut(
    resumen_gasto_usuario_completo["yearly_income"],
    q=4,
    labels=["Bajo", "Medio-bajo", "Medio-alto", "Alto"]
)

# Vemos los rangos de cada segmento y cuántos usuarios caen en cada uno
print(resumen_gasto_usuario_completo.groupby("segmento_ingresos")["yearly_income"].agg(["min", "max", "count"]))

                       min       max  count
segmento_ingresos                          
Bajo                   1.0   32817.0    500
Medio-bajo         32819.0   40744.0    500
Medio-alto         40745.0   52680.0    500
Alto               52754.0  307018.0    500


##### Los 4 segmentos quedaron perfectamente balanceados (500 usuarios cada uno), con rangos que tienen sentido de negocio:

Bajo: 1 – 32.817 €

Medio-bajo: 32.819 – 40.744 €

Medio-alto: 40.745 – 52.680 €

Alto: 52.754 – 307.018 €


Nota el salto grande en "Alto" (hasta 307.018 €) — es normal en datos de ingresos, suele haber una cola larga de valores altos que estira el máximo del último cuartil, sin que eso afecte a que los otros tres estén bien repartidos.

##### Paso 2 — Analizar el patrón de gasto por segmento de ingresos

Ahora cruzamos el segmento con las métricas de gasto que ya calculamos, para ver si el patrón de consumo varía según el nivel de ingresos:

In [29]:
patron_gasto_por_segmento = resumen_gasto_usuario_completo.groupby("segmento_ingresos").agg(
    num_usuarios=("id", "count"),
    gasto_neto_medio=("gasto_neto_total", "mean"),
    num_transacciones_medio=("num_transacciones", "mean"),
    ticket_medio_promedio=("ticket_medio", "mean"),
    deuda_media=("total_debt", "mean"),
    credit_score_medio=("credit_score", "mean")
).reset_index()

print(patron_gasto_por_segmento)

  segmento_ingresos  num_usuarios  gasto_neto_medio  num_transacciones_medio  \
0              Bajo           500      229943.46830                 6659.128   
1        Medio-bajo           500      246718.64246                 6376.488   
2        Medio-alto           500      268696.44962                 6405.220   
3              Alto           500      398312.48418                 7170.994   

   ticket_medio_promedio  deuda_media  credit_score_medio  
0              23.475306    33045.324             708.464  
1              24.061762    52231.572             711.432  
2              25.456201    68603.810             708.870  
3              34.056777   100958.070             710.172  


##### Resultado interesante, con un patrón bastante claro:

El gasto neto crece con los ingresos, de forma bastante progresiva (229.943 € → 398.312 € del segmento Bajo al Alto), lo cual es esperable.

El ticket medio también sube con los ingresos (23,5 € → 34 €), pero de forma menos pronunciada que el gasto total — sugiere que los usuarios de mayores ingresos no solo gastan más por transacción, sino que además transaccionan más veces.

El número de transacciones no sigue un patrón lineal claro: el segmento "Medio-alto" tiene el número más bajo (6.405), mientras que "Alto" vuelve a subir (7.171) — no es una relación perfecta entre ingresos y frecuencia de uso.

La deuda media crece de forma mucho más pronunciada que los ingresos (33.045 € → 100.958 €, prácticamente x3), mientras que el credit_score se mantiene muy estable entre segmentos (708-711) — esto es un hallazgo relevante: tener más ingresos no está correlacionado con mejor score crediticio en este dataset, aunque sí con más deuda absoluta (lo cual tiene sentido, mayor capacidad de endeudamiento).

Con esto cerramos el análisis del Bloque 2. Tenemos 3 tablas resumen listas:

resumen_gasto_usuario_completo — gasto por usuario (2.000 filas)

gasto_usuario_categoria — gasto por usuario y categoría (93.238 filas)

patron_gasto_por_segmento — gasto agregado por segmento de ingresos (4 filas)

#### Bloque 3: Análisis por tarjeta

Cubre tres cosas: uso respecto al límite de crédito, comparativa por tipo de tarjeta, y antigüedad vs. actividad.

##### Paso 1 — Gasto y uso por tarjeta, comparado con su límite de crédito

Igual que hicimos a nivel usuario, ahora agregamos las transacciones a nivel de tarjeta (card_id), y lo cruzamos con credit_limit para ver qué proporción de su límite está usando cada tarjeta.

##### Uso how="left" desde cards (no desde el resultado del groupby) por la misma razón que con los usuarios: queremos ver todas las tarjetas, incluidas las que no tienen ninguna transacción registrada, no solo las activas.

In [30]:
resumen_gasto_tarjeta = trans_completo.groupby("card_id").agg(
    gasto_neto_total=("amount", "sum"),
    num_transacciones=("amount", "count"),
    ticket_medio=("amount", "mean")
).reset_index()

# Unimos con los datos propios de cada tarjeta (límite, marca, tipo, fechas)
resumen_gasto_tarjeta = cards[["id", "client_id", "card_brand", "card_type", "credit_limit", "acct_open_date", "expires", "has_chip"]].merge(
    resumen_gasto_tarjeta,
    left_on="id", right_on="card_id",
    how="left"
)

# Rellenamos con 0 las tarjetas sin transacciones (mismo caso que con usuarios)
columnas_a_rellenar = ["gasto_neto_total", "num_transacciones", "ticket_medio"]
resumen_gasto_tarjeta[columnas_a_rellenar] = resumen_gasto_tarjeta[columnas_a_rellenar].fillna(0)

# Calculamos el % de uso del límite (usando el gasto absoluto, no el neto, para no distorsionar con devoluciones)
resumen_gasto_tarjeta["pct_uso_limite"] = (resumen_gasto_tarjeta["gasto_neto_total"] / resumen_gasto_tarjeta["credit_limit"]) * 100

resumen_gasto_tarjeta = resumen_gasto_tarjeta.drop(columns=["card_id"])

print(resumen_gasto_tarjeta.shape)
print(resumen_gasto_tarjeta[["id", "credit_limit", "gasto_neto_total", "pct_uso_limite"]].head(5))

(6146, 12)
     id  credit_limit  gasto_neto_total  pct_uso_limite
0  4524       24295.0         162494.10      668.837621
1  2731       21968.0          74413.26      338.734796
2  3701       46414.0         151411.18      326.218770
3    42       12400.0         524079.76     4226.449677
4  4659          28.0              0.00        0.000000


##### El .shape confirma las 6.146 filas esperadas, pero hay un problema conceptual importante en pct_uso_limite: los valores no tienen sentido como "porcentaje de uso del límite" — 668%, 338%, hasta 4226% en la fila 3.

La causa: gasto_neto_total es la suma de todas las transacciones acumuladas en todo el periodo del dataset (varios años, según vimos en transactions), mientras que credit_limit es un límite que se aplica por ciclo de facturación (normalmente mensual), no un tope acumulado para toda la vida de la tarjeta. Estamos comparando una magnitud acumulada contra un límite periódico — es como comparar el gasto total de 5 años contra el sueldo de un mes.

Cómo lo corregimos: para que el ratio tenga sentido de negocio, hay que comparar el límite contra el gasto medio mensual de la tarjeta, no contra el acumulado total.

##### .clip(lower=1) evita que una tarjeta abierta el mismo mes que la fecha máxima del dataset divida por 0 meses.

In [31]:
# Calculamos el número de meses activos de cada tarjeta (desde apertura hasta la fecha máxima del dataset, o hasta expiración si es antes)
fecha_max_dataset = transactions["date"].max()

resumen_gasto_tarjeta["meses_activa"] = (
    (fecha_max_dataset.year - resumen_gasto_tarjeta["acct_open_date"].dt.year) * 12 +
    (fecha_max_dataset.month - resumen_gasto_tarjeta["acct_open_date"].dt.month)
).clip(lower=1)  # mínimo 1 mes, para evitar división por 0

# Gasto medio mensual y ratio corregido frente al límite
resumen_gasto_tarjeta["gasto_medio_mensual"] = resumen_gasto_tarjeta["gasto_neto_total"] / resumen_gasto_tarjeta["meses_activa"]
resumen_gasto_tarjeta["pct_uso_limite_mensual"] = (resumen_gasto_tarjeta["gasto_medio_mensual"] / resumen_gasto_tarjeta["credit_limit"]) * 100

print(resumen_gasto_tarjeta[["id", "credit_limit", "meses_activa", "gasto_medio_mensual", "pct_uso_limite_mensual"]].head(5))

     id  credit_limit  meses_activa  gasto_medio_mensual  \
0  4524       24295.0           205           792.654146   
1  2731       21968.0            66          1127.473636   
2  3701       46414.0           195           776.467590   
3    42       12400.0           201          2607.361990   
4  4659          28.0           133             0.000000   

   pct_uso_limite_mensual  
0                3.262623  
1                5.132345  
2                1.672917  
3               21.027113  
4                0.000000  


##### Mucho más razonable ahora: valores entre 0% y 21% en esta muestra, en línea con lo que se esperaría de un ratio de uso mensual del límite de crédito.

Antes de dar esto por bueno del todo, quiero que revisemos un caso concreto que llama la atención: la fila 4 (id 4659) tiene un credit_limit de solo 28,0 € y gasto_medio_mensual = 0. 
Un límite de crédito de 28 € es extremadamente bajo para una tarjeta real — podría ser un dato atípico genuino (tarjeta de prueba, tarjeta prácticamente inactiva) o un error de captura en el dataset original. Vale la pena comprobar si hay más casos así antes de seguir, porque estos límites tan bajos pueden distorsionar el ratio para arriba de forma artificial en cuanto tengan algo de gasto.

In [32]:
# Revisamos la distribución de credit_limit para detectar valores atípicamente bajos
print(resumen_gasto_tarjeta["credit_limit"].describe())
print("\nTarjetas con límite menor a 100€:", (resumen_gasto_tarjeta["credit_limit"] < 100).sum())

count      6146.000000
mean      14347.493980
std       12014.463884
min           0.000000
25%        7042.750000
50%       12592.500000
75%       19156.500000
max      151223.000000
Name: credit_limit, dtype: float64

Tarjetas con límite menor a 100€: 575


##### Ahí está el dato que faltaba: 575 tarjetas con límite menor a 100€ (un 9,4% del total) — no es un caso aislado, es un patrón con peso real que merece explicación antes de seguir, igual que hicimos con los nulos de merchant_state.

Aún no tengo el resultado del segundo bloque de código (tarjetas con credit_limit == 0 específicamente, cruzado con card_type y card_brand) — ese es el que nos va a decir si el patrón se explica por tipo de tarjeta (por ejemplo, es razonable que las tarjetas de débito tengan límites bajos o incluso 0, porque no funcionan como crédito real, mientras que sí sería raro en una tarjeta de crédito).

In [33]:
tarjetas_limite_cero = resumen_gasto_tarjeta[resumen_gasto_tarjeta["credit_limit"] == 0]
print("Tarjetas con límite = 0:", len(tarjetas_limite_cero))
print(tarjetas_limite_cero[["id", "client_id", "card_brand", "card_type", "gasto_neto_total", "pct_uso_limite_mensual"]])

# Y lo más importante: ¿el patrón de límite bajo se concentra en un tipo de tarjeta concreto?
print("\nDistribución de card_type para límite < 100€:")
print(resumen_gasto_tarjeta[resumen_gasto_tarjeta["credit_limit"] < 100]["card_type"].value_counts())

Tarjetas con límite = 0: 31
        id  client_id  card_brand        card_type  gasto_neto_total  \
221   4318        668  Mastercard           Credit         137691.71   
293   3626        870        Amex           Credit          40006.06   
478   5957       1975  Mastercard           Credit         127228.86   
801   1799        934        Visa           Credit              0.00   
1012   782       1658        Visa           Credit              0.00   
1340    96         81  Mastercard           Credit         104611.54   
1501  5680        214        Visa           Credit              0.00   
1621  3443        846        Visa           Credit         265431.80   
1823  5264       1770        Visa           Credit          93215.06   
2084  1577         93  Mastercard  Debit (Prepaid)              0.00   
2288  5781       1942  Mastercard            Debit           1734.68   
2405  3182         15        Visa           Credit              0.00   
2568  3183        318        Visa   

##### Este resultado matiza bastante lo que pensábamos. Dos hallazgos distintos que hay que separar:

1. El patrón general (límite < 100€) sí tiene explicación de negocio: de las 575 tarjetas con límite bajo, 537 son "Debit (Prepaid)" — coherente, una prepago no tiene un "límite de crédito" real, tiene el saldo que el usuario carga. Esto no es un error.
   
2. Pero el caso de credit_limit == 0 específicamente es distinto y sí es un problema real: de las 31 tarjetas con límite exactamente 0, la mayoría son de tipo "Credit" (Mastercard, Visa, Amex, Discover) — y ahí sí hay algo raro, porque una tarjeta de crédito con límite 0 no tiene sentido operativo (¿cómo se autorizaron 137.691 € en gasto acumulado, como en la fila 221, con límite 0?). Esto ya no se explica por el tipo de tarjeta, es una inconsistencia del dato de origen, similar a lo que vimos con use_chip/merchant_city.

   
3. El error técnico visible: pct_uso_limite_mensual da inf (cuando hay gasto y límite es 0, división por cero) o NaN (cuando además el gasto es 0, división 0/0) — hay que corregir esto antes de que contamine cualquier media o gráfico posterior.

   
Cómo lo resolvemos, con dos ajustes distintos para dos problemas distintos:

In [34]:
# 1. Para tarjetas prepago, el concepto "% uso de límite de crédito" no aplica de la misma forma
#    Las dejamos identificadas pero no las tratamos como anómalas
print("Tipos de tarjeta en el dataset completo:")
print(resumen_gasto_tarjeta["card_type"].value_counts())

# 2. Para credit_limit == 0 con card_type == "Credit", es una inconsistencia real -> las marcamos aparte
resumen_gasto_tarjeta["limite_inconsistente"] = (
    (resumen_gasto_tarjeta["credit_limit"] == 0) & (resumen_gasto_tarjeta["card_type"] == "Credit")
)
print("\nTarjetas de crédito con límite inconsistente (0€):", resumen_gasto_tarjeta["limite_inconsistente"].sum())

# 3. Corregimos el ratio: si credit_limit es 0, el ratio queda como NaN explícito (no inf), documentado como "no calculable"
resumen_gasto_tarjeta["pct_uso_limite_mensual"] = resumen_gasto_tarjeta.apply(
    lambda row: (row["gasto_medio_mensual"] / row["credit_limit"] * 100) if row["credit_limit"] > 0 else np.nan,
    axis=1
)

print("\nValores infinitos restantes:", np.isinf(resumen_gasto_tarjeta["pct_uso_limite_mensual"]).sum())
print("Valores NaN (no calculables):", resumen_gasto_tarjeta["pct_uso_limite_mensual"].isnull().sum())

Tipos de tarjeta en el dataset completo:
card_type
Debit              3511
Credit             2057
Debit (Prepaid)     578
Name: count, dtype: int64

Tarjetas de crédito con límite inconsistente (0€): 26

Valores infinitos restantes: 0
Valores NaN (no calculables): 31


##### Todo correcto: 0 valores infinitos restantes, y 31 NaN documentados como "no calculables" (los 26 casos de crédito inconsistente + los 5 casos de débito/prepago que también tenían límite 0, que sí es coherente para ese tipo de tarjeta).

Con esto, pct_uso_limite_mensual queda limpio y utilizable para cualquier media, gráfico o tabla dinámica sin riesgo de que un inf reviente los cálculos.

Resumen de lo que hemos resuelto en este punto:

3.511 Debit + 578 Debit (Prepaid) + 2.057 Credit = 6.146 tarjetas, confirma que no hay tipos sueltos sin clasificar.

26 tarjetas de crédito con límite inconsistente (0€) quedan marcadas explícitamente con limite_inconsistente = True, documentadas como anomalía del dato de origen, sin inventar un valor que no tenemos forma de conocer.

El ratio de uso de límite ya es fiable para el resto de la base.

##### Punto 2: comparativa por tipo de tarjeta

Ya tenemos casi todo lo necesario en resumen_gasto_tarjeta, solo hay que agregar por card_type.

##### Un par de notas sobre las métricas elegidas:

credit_limit_medio y pct_uso_limite_medio van a tener menos sentido comparativo para "Debit" (sin límite de crédito real como tal, aunque el dataset les asigne un valor), así que interprétalo con cautela para ese tipo — pero no lo excluyo del cálculo, mejor verlo y decidir juntos si hace falta filtrarlo.

pct_uso_limite_medio ya no arrastra los inf/NaN problemáticos gracias a la limpieza que hicimos antes.

In [35]:
comparativa_tipo_tarjeta = resumen_gasto_tarjeta.groupby("card_type").agg(
    num_tarjetas=("id", "count"),
    gasto_neto_total=("gasto_neto_total", "sum"),
    gasto_medio_mensual_promedio=("gasto_medio_mensual", "mean"),
    num_transacciones_total=("num_transacciones", "sum"),
    ticket_medio_promedio=("ticket_medio", "mean"),
    credit_limit_medio=("credit_limit", "mean"),
    pct_uso_limite_medio=("pct_uso_limite_mensual", "mean")
).reset_index()

print(comparativa_tipo_tarjeta)

         card_type  num_tarjetas  gasto_neto_total  \
0           Credit          2057      2.258118e+08   
1            Debit          3511      3.260915e+08   
2  Debit (Prepaid)           578      1.993220e+07   

   gasto_medio_mensual_promedio  num_transacciones_total  \
0                    897.922074                4109189.0   
1                    768.505271                8280996.0   
2                    296.526977                 915730.0   

   ticket_medio_promedio  credit_limit_medio  pct_uso_limite_medio  
0              37.815258        11174.380165             10.632429  
1              26.222258        18557.888636             10.850830  
2              12.983938           64.448097            478.042448  


##### Muy revelador este resultado, con un problema estadístico claro en la última columna: pct_uso_limite_medio para "Debit (Prepaid)" es 478% — un valor descontrolado comparado con el 10,6% y 10,8% de Credit y Debit.

La causa: ya vimos que las prepago tienen un credit_limit_medio muy bajo (64,45 €) — con denominadores tan pequeños, cualquier tarjeta prepago con algo de gasto genera un ratio individual enorme (ej. gastar 200€/mes con un límite de 64€ da 312%), y al hacer la media aritmética, unos pocos casos extremos arrastran el promedio hacia arriba de forma desproporcionada. La media es sensible a outliers; en este caso concreto nos está mintiendo sobre el "usuario típico".

Solución: usar la mediana en vez de la media para esta métrica en concreto — la mediana no se deja arrastrar por valores extremos, y nos daría una imagen más realista del caso típico.

In [36]:
# Recalculamos con mediana además de media, para comparar
comparativa_tipo_tarjeta_v2 = resumen_gasto_tarjeta.groupby("card_type").agg(
    num_tarjetas=("id", "count"),
    gasto_neto_total=("gasto_neto_total", "sum"),
    gasto_medio_mensual_promedio=("gasto_medio_mensual", "mean"),
    ticket_medio_promedio=("ticket_medio", "mean"),
    credit_limit_medio=("credit_limit", "mean"),
    pct_uso_limite_media=("pct_uso_limite_mensual", "mean"),
    pct_uso_limite_mediana=("pct_uso_limite_mensual", "median")
).reset_index()

print(comparativa_tipo_tarjeta_v2[["card_type", "credit_limit_medio", "pct_uso_limite_media", "pct_uso_limite_mediana"]])

         card_type  credit_limit_medio  pct_uso_limite_media  \
0           Credit        11174.380165             10.632429   
1            Debit        18557.888636             10.850830   
2  Debit (Prepaid)           64.448097            478.042448   

   pct_uso_limite_mediana  
0                5.461613  
1                2.879095  
2              254.547314  


##### La mediana baja bastante respecto a la media (478% → 254%), así que sí había algo de efecto outlier, pero el patrón de fondo se mantiene: incluso el caso "típico" de una tarjeta prepago gasta 2,5 veces su credit_limit registrado cada mes. Esto ya no se explica por unos pocos casos extremos, es sistemático en todo el grupo.

Conclusión de negocio: para las tarjetas prepago, el campo credit_limit no funciona como un límite de crédito real (que no debería poder superarse), sino más bien como una carga inicial o un valor de referencia técnico — el usuario puede recargar el saldo repetidamente, por eso el gasto mensual acumulado supera con normalidad ese número de forma constante. Es decir: la métrica pct_uso_limite no es comparable entre tipos de tarjeta — tiene sentido real para "Credit" (donde sí existe un tope que no se debe superar), pero no para "Debit (Prepaid)" (donde el número de referencia no actúa como techo).

Cómo lo dejamos documentado: en vez de forzar una comparación que confunde, lo más honesto es excluir explícitamente "Debit (Prepaid)" de esa columna concreta en la tabla final, dejando una nota clara del motivo — así en Excel no habrá una cifra rara sin explicación.

In [37]:
# Marcamos el % de uso de límite como "No aplicable" para tarjetas prepago, ya que el campo no representa un tope real
comparativa_tipo_tarjeta_v2.loc[
    comparativa_tipo_tarjeta_v2["card_type"] == "Debit (Prepaid)",
    ["pct_uso_limite_media", "pct_uso_limite_mediana"]
] = np.nan

print(comparativa_tipo_tarjeta_v2)

         card_type  num_tarjetas  gasto_neto_total  \
0           Credit          2057      2.258118e+08   
1            Debit          3511      3.260915e+08   
2  Debit (Prepaid)           578      1.993220e+07   

   gasto_medio_mensual_promedio  ticket_medio_promedio  credit_limit_medio  \
0                    897.922074              37.815258        11174.380165   
1                    768.505271              26.222258        18557.888636   
2                    296.526977              12.983938           64.448097   

   pct_uso_limite_media  pct_uso_limite_mediana  
0             10.632429                5.461613  
1             10.850830                2.879095  
2                   NaN                     NaN  


##### Perfecto, la tabla queda exactamente como buscábamos: Credit y Debit con valores comparables (10,6% y 10,9% de media), y Debit (Prepaid) correctamente marcado como NaN en vez de mostrar un 478% engañoso.

Con esto cerramos el punto 2 del Bloque 3 (comparativa por tipo de tarjeta). Resumen de lo que muestra:

Debit es el tipo más numeroso (3.511 tarjetas) y con mayor gasto neto total agregado (326M€), aunque no es el que más gasta por tarjeta individualmente.

Credit tiene el gasto medio mensual más alto por tarjeta (897,92€) y el ticket medio más alto (37,82€) — coherente con que el crédito suele usarse para compras de mayor importe.

Debit (Prepaid) es claramente el uso más ligero, tanto en volumen de tarjetas como en gasto por tarjeta.

##### Punto 3: antigüedad de la tarjeta vs. actividad. 
Ya tenemos meses_activa calculado desde antes, así que solo falta ver si hay correlación entre cuánto tiempo lleva abierta una tarjeta y su nivel de actividad (gasto, número de transacciones).

pct_tarjetas_inactivas nos dice qué porcentaje de tarjetas de cada rango de antigüedad no ha tenido ninguna transacción — útil para ver si las tarjetas más nuevas o más viejas tienden más a estar inactivas.

In [38]:
# Creamos rangos de antigüedad para poder comparar grupos, en vez de una correlación pura
resumen_gasto_tarjeta["rango_antiguedad"] = pd.cut(
    resumen_gasto_tarjeta["meses_activa"],
    bins=[0, 24, 60, 120, 300],
    labels=["0-2 años", "2-5 años", "5-10 años", "10+ años"]
)

antiguedad_vs_actividad = resumen_gasto_tarjeta.groupby("rango_antiguedad").agg(
    num_tarjetas=("id", "count"),
    gasto_medio_mensual_promedio=("gasto_medio_mensual", "mean"),
    num_transacciones_promedio=("num_transacciones", "mean"),
    pct_tarjetas_inactivas=("num_transacciones", lambda x: (x == 0).mean() * 100)
).reset_index()

print(antiguedad_vs_actividad)

  rango_antiguedad  num_tarjetas  gasto_medio_mensual_promedio  \
0         0-2 años          1384                    140.735719   
1         2-5 años           386                   1058.639952   
2        5-10 años          1527                   1182.182105   
3         10+ años          2833                    812.178260   

   num_transacciones_promedio  pct_tarjetas_inactivas  
0                   37.282514               90.245665  
1                 1188.626943               19.170984  
2                 2654.863785               14.341847  
3                 3069.827391               18.708083  


##### Un patrón muy claro y con sentido de negocio: cuanto más nueva es la tarjeta, mayor la probabilidad de estar inactiva. El salto es enorme entre el primer grupo y el resto:

0-2 años: 90,2% de tarjetas inactivas, gasto mensual medio muy bajo (140,74€) — coherente con tarjetas recién emitidas que aún no han empezado a usarse, o que el cliente solicitó pero no activó.

2-5 años: cae drásticamente a 19,2% inactivas, con gasto ya mucho mayor (1.058,64€).

5-10 años: sigue bajando la inactividad (14,3%) y el gasto mensual es el más alto de todos los grupos (1.182,18€) — el "pico" de actividad parece estar aquí.

10+ años: la inactividad sube ligeramente (18,7%) y el gasto baja un poco (812,18€) respecto al grupo anterior — podría sugerir cierto declive de uso en tarjetas muy antiguas, aunque siguen siendo bastante activas.

Esto es un hallazgo genuino y útil: sugiere que hay un periodo de "arranque" en los primeros 2 años donde la tarjeta tiene alta probabilidad de no usarse, y que la actividad se consolida a partir del segundo año. Para negocio, esto podría alimentar una estrategia de activación temprana (contactar/incentivar a clientes con tarjetas nuevas sin uso).

#### Bloque 4: Análisis temporal

Cubre dos cosas: evolución mensual del volumen/importe de transacciones, y detección de estacionalidad.

##### Paso 1 — Evolución mensual de transacciones e importe Paso 1 — Evolución mensual de transacciones e importe

##### Primero necesitamos extraer el mes-año de cada transacción (ya tenemos date como datetime, así que es sencillo), y agregar por ese periodo.

Uso .dt.to_period("M") en vez de agrupar directamente por fecha completa, porque redondea automáticamente cualquier fecha al mes-año al que pertenece (ej. 2010-01-15 y 2010-01-28 caen ambas en 2010-01), que es justo el nivel de agregación que buscamos.

Miro tanto .head() como .tail() porque, dado que vimos rangos de "meses relativos" desde -60 hasta 0 en tu informe de Power BI, es útil confirmar aquí también el rango real de fechas (primer y último mes) antes de interpretar cualquier tendencia.

In [39]:
# Creamos una columna de periodo mensual (año-mes) a partir de la fecha
trans_completo["periodo_mes"] = trans_completo["date"].dt.to_period("M")

evolucion_mensual = trans_completo.groupby("periodo_mes").agg(
    num_transacciones=("amount", "count"),
    importe_total=("amount", "sum"),
    ticket_medio=("amount", "mean")
).reset_index()

# Convertimos el periodo a string para que se exporte bien a Excel más adelante
evolucion_mensual["periodo_mes"] = evolucion_mensual["periodo_mes"].astype(str)

print(evolucion_mensual.shape)
print(evolucion_mensual.head(10))
print(evolucion_mensual.tail(10))

(118, 4)
  periodo_mes  num_transacciones  importe_total  ticket_medio
0     2010-01             101209     4372532.12     43.202997
1     2010-02              93470     4103170.24     43.898259
2     2010-03             103345     4539853.38     43.929105
3     2010-04             100169     4407951.33     44.005145
4     2010-05             104773     4610601.80     44.005629
5     2010-06             102677     4509874.37     43.922927
6     2010-07             106034     4660340.55     43.951379
7     2010-08             107547     4688090.24     43.591083
8     2010-09             103902     4540028.90     43.695298
9     2010-10             106150     4623129.08     43.552794
    periodo_mes  num_transacciones  importe_total  ticket_medio
108     2019-01             118116     5018161.92     42.485031
109     2019-02             106385     4539678.49     42.672167
110     2019-03             118548     5077167.67     42.827949
111     2019-04             114034     4918536.08    

##### Buen resultado: 118 meses de datos, desde enero 2010 hasta octubre 2019 — casi 10 años completos de historial. Esto confirma la escala temporal real del dataset (coherente con lo que vimos antes en el rango "-60 a 0" de la página 3 del informe anterior, aunque aquí lo tenemos en fechas absolutas en vez de relativas).

Los números se ven estables y coherentes: el volumen de transacciones ronda 100.000-118.000 por mes en todo el periodo, sin caídas ni picos extraños a primera vista, y el ticket medio se mantiene muy constante (42-44€) durante los casi 10 años — buena señal de consistencia en los datos.

##### Paso 2 — Detección de estacionalidad

Ahora extraemos el mes del año (independientemente del año) para ver si hay patrones que se repiten cada año — por ejemplo, más gasto en diciembre (Navidad) o menos en verano.

In [40]:
# Extraemos el número de mes (1-12) de cada transacción, ignorando el año
trans_completo["mes_del_anio"] = trans_completo["date"].dt.month

estacionalidad = trans_completo.groupby("mes_del_anio").agg(
    num_transacciones_medio=("amount", "count"),
    importe_medio_total=("amount", "sum"),
    ticket_medio=("amount", "mean")
).reset_index()

# Como agregamos varios años juntos, dividimos por el número de años presentes para tener un promedio anual comparable
num_anios = trans_completo["date"].dt.year.nunique()
estacionalidad["num_transacciones_medio"] = estacionalidad["num_transacciones_medio"] / num_anios
estacionalidad["importe_medio_total"] = estacionalidad["importe_medio_total"] / num_anios

print(estacionalidad)

    mes_del_anio  num_transacciones_medio  importe_medio_total  ticket_medio
0              1                 113915.5          4870615.719     42.756392
1              2                 103135.1          4452296.071     43.169552
2              3                 114539.0          4931464.753     43.054896
3              4                 110618.2          4779747.855     43.209416
4              5                 114619.4          4919244.078     42.918076
5              6                 111852.2          4842479.772     43.293559
6              7                 115367.5          4969631.504     43.076529
7              8                 115687.3          4951032.078     42.796678
8              9                 111779.5          4794495.474     42.892440
9             10                 114863.8          4943576.438     43.038594
10            11                 100348.8          4290083.701     42.751719
11            12                 103865.2          4438884.785     42.736978

##### El resultado muestra una estacionalidad bastante suave, sin picos ni caídas dramáticas como cabría esperar en un patrón de consumo típico (por ejemplo, no hay un salto claro en diciembre por Navidad):

Mes con más transacciones: agosto (115.687), seguido de cerca por marzo, junio y julio — todos en el rango 114.000-115.700.
Mes con menos transacciones: noviembre (100.349), seguido de febrero (103.135) y diciembre (103.865).

El ticket medio es prácticamente plano todo el año (42,7€ - 44,2€), la variación de mayor a menor apenas supera 1,5€ — el importe medio por transacción no cambia por temporada, solo cambia ligeramente el volumen de transacciones.

Lectura de negocio: este dataset no muestra un patrón estacional fuerte tipo "picos de Navidad" o "bajón de verano" que uno esperaría en datos reales de consumo — la variación entre el mes más alto (agosto) y el más bajo (noviembre) es de apenas un 15% en volumen, y casi nula en importe medio.

Esto puede deberse a que es un dataset sintético/simulado (recordemos que muchos datasets de este tipo en Kaggle son generados artificialmente para pruebas), lo cual explicaría por qué no replica fielmente los patrones estacionales de un banco real.

#### Bloque 5: Análisis de fraude

##### Paso 1 — Cargar y explorar el archivo de etiquetas de fraude

Igual que hicimos con mcc_codes.json, primero cargamos el JSON y revisamos su estructura antes de mapear nada, para no repetir el error de tipos que tuvimos con card_on_dark_web.

Este archivo puede venir estructurado de dos formas típicas: como un diccionario {"target": {"id_transaccion": "Yes"/"No", ...}} (formato habitual en este dataset de Kaggle) o como una lista plana de valores. Por eso primero comprobamos type() antes de asumir nada.

In [41]:
with open(os.path.join(ruta_datos, "train_fraud_labels.json"), "r") as f:
    fraud_labels = json.load(f)

# Vemos la estructura general del JSON
print(type(fraud_labels))
print(list(fraud_labels.keys())[:3] if isinstance(fraud_labels, dict) else fraud_labels[:3])

<class 'dict'>
['target']


##### Confirmado: es un diccionario con una única clave "target" — esto significa que la estructura real es {"target": {"id_transaccion": "Yes"/"No", ...}}, tal como sospechaba. Necesitamos entrar un nivel más para ver los datos reales.

##### necesitamos confirmar dos cosas antes de mapear: si las claves son el id de la transacción (como string o int), y qué valores usa para marcar fraude ("Yes"/"No", 1/0, True/False, etc.), para hacer el mapeo correctamente a la primera.

In [42]:
# Entramos al diccionario interno "target" para ver su estructura
fraud_target = fraud_labels["target"]
print(type(fraud_target))
print(len(fraud_target))

# Vemos las primeras 5 claves-valor para confirmar el formato exacto
print(list(fraud_target.items())[:5])

<class 'dict'>
8914963
[('10649266', 'No'), ('23410063', 'No'), ('9316588', 'No'), ('12478022', 'No'), ('9558530', 'No')]


#####  Confirmado el formato: diccionario con clave = id de transacción como string, valor = 'Yes'/'No'. 
Pero hay un dato importante: len(fraud_target) es 8.914.963, mientras que transactions tiene 13.305.915 filas — es decir, solo el 67% de las transacciones tienen etiqueta de fraude. 
Esto es coherente con el nombre del archivo (train_fraud_labels, sugiere que es un subconjunto de entrenamiento, no la totalidad), pero hay que confirmarlo antes de mapear para no interpretar mal los huecos después.

In [43]:
# Mapeamos el fraude, convirtiendo el id de transacción a string para que coincida con las claves del diccionario
trans_completo["es_fraude"] = trans_completo["id"].astype(str).map(fraud_target)

# Confirmamos cobertura del mapeo
print("Transacciones con etiqueta:", trans_completo["es_fraude"].notnull().sum())
print("Transacciones sin etiqueta:", trans_completo["es_fraude"].isnull().sum())
print("\nDistribución de valores:")
print(trans_completo["es_fraude"].value_counts())

Transacciones con etiqueta: 8914963
Transacciones sin etiqueta: 4390952

Distribución de valores:
es_fraude
No     8901631
Yes      13332
Name: count, dtype: int64


##### Confirmado y coherente: 8.914.963 con etiqueta (coincide exactamente con el tamaño del diccionario, mapeo perfecto sin pérdidas) y 4.390.952 sin etiqueta (las que no forman parte del subconjunto de entrenamiento).

De las que sí tienen etiqueta: 13.332 son fraude (Yes) frente a 8.901.631 legítimas (No) — una tasa de fraude del 0,15%, típico de este tipo de problemas (el fraude real es siempre una minoría muy pequeña, es lo que hace difícil detectarlo).

##### Paso 2 — Comparar el perfil de transacciones fraudulentas vs. legítimas

Ahora construimos la tabla comparativa que planeamos: importe medio, categoría de comercio y hora del día, comparando fraude vs. no fraude. 
Trabajamos solo con las transacciones que sí tienen etiqueta (excluyendo los NaN, que no podemos clasificar).

In [44]:
# Filtramos solo transacciones con etiqueta conocida
trans_con_etiqueta = trans_completo[trans_completo["es_fraude"].notnull()].copy()

# Extraemos la hora del día para el análisis
trans_con_etiqueta["hora_del_dia"] = trans_con_etiqueta["date"].dt.hour

# Comparativa general: importe medio y tipo de transacción
comparativa_fraude = trans_con_etiqueta.groupby("es_fraude").agg(
    num_transacciones=("id", "count"),
    importe_medio=("amount", "mean"),
    importe_mediana=("amount", "median"),
    hora_media=("hora_del_dia", "mean")
).reset_index()

print(comparativa_fraude)

  es_fraude  num_transacciones  importe_medio  importe_mediana  hora_media
0        No            8901631      42.848614           28.950   12.389884
1       Yes              13332     110.234682           69.975   12.008251


##### Diferencia clara en el importe: las transacciones fraudulentas tienen un importe medio 2,6 veces mayor que las legítimas (110,23€ vs. 42,85€), y lo mismo con la mediana (69,98€ vs. 28,95€) — el fraude tiende a ir por importes más altos, patrón coherente con lo que se suele ver en detección de fraude real (los defraudadores buscan maximizar el beneficio por transacción antes de ser detectados).

En cambio, la hora media es casi idéntica (12,39 vs. 12,09, ambas cerca del mediodía) — no hay un patrón horario diferenciador claro con esta métrica agregada. Ojo: la media de horas puede ocultar patrones reales (por ejemplo, si el fraude se concentra tanto de madrugada como a mediodía, la media sale "normal" aunque el patrón real no lo sea) — así que antes de descartar el horario como factor, conviene mirarlo con una distribución por franjas, no solo la media.

In [45]:
# Comparamos la distribución de fraude por franja horaria, no solo la media
trans_con_etiqueta["franja_horaria"] = pd.cut(
    trans_con_etiqueta["hora_del_dia"],
    bins=[0, 6, 12, 18, 24],
    labels=["Madrugada (0-6h)", "Mañana (6-12h)", "Tarde (12-18h)", "Noche (18-24h)"],
    include_lowest=True
)

fraude_por_franja = trans_con_etiqueta.groupby("franja_horaria")["es_fraude"].apply(
    lambda x: (x == "Yes").mean() * 100
).reset_index(name="pct_fraude")

print(fraude_por_franja)

     franja_horaria  pct_fraude
0  Madrugada (0-6h)    0.101179
1    Mañana (6-12h)    0.176785
2    Tarde (12-18h)    0.177849
3    Noche (18-24h)    0.043065


##### Y también la categoría de comercio, el otro factor que planeamos:

In [46]:
# Top 10 categorías con mayor tasa de fraude (con un mínimo de transacciones para evitar ruido estadístico)
fraude_por_categoria = trans_con_etiqueta.groupby("mcc_categoria").agg(
    num_transacciones=("es_fraude", "count"),
    pct_fraude=("es_fraude", lambda x: (x == "Yes").mean() * 100)
).reset_index()

fraude_por_categoria = fraude_por_categoria[fraude_por_categoria["num_transacciones"] >= 100].sort_values("pct_fraude", ascending=False)
print(fraude_por_categoria.head(10))

                                        mcc_categoria  num_transacciones  \
24                                       Cruise Lines                276   
73                 Music Stores - Musical Instruments                204   
63            Miscellaneous Fabricated Metal Products                245   
22           Computers, Computer Peripheral Equipment               1883   
40                              Floor Covering Stores                222   
67                    Miscellaneous Metal Fabrication                256   
35                                 Electronics Stores               4689   
37               Fabricated Structural Metal Products                273   
82                         Precious Stones and Metals               3525   
42  Furniture, Home Furnishings, and Equipment Stores               2600   

    pct_fraude  
24   59.782609  
73   37.254902  
63   11.836735  
22   10.833776  
40   10.360360  
67    8.593750  
35    8.573257  
37    8.058608  
82    6.86

##### Dos hallazgos muy claros:

Por franja horaria: contrario a la intuición típica de "el fraude ocurre de madrugada", aquí el patrón es el opuesto — Mañana y Tarde tienen la tasa más alta (0,18% ambas), mientras que Noche es la más baja (0,04%). Madrugada queda en un nivel intermedio (0,10%). Esto tiene sentido si el fraude se camufla entre el volumen alto de transacciones legítimas del día, en vez de destacar en horas de menor actividad donde sería más fácil detectarlo.

Por categoría de comercio: aquí hay un hallazgo mucho más contundente — "Cruise Lines" tiene una tasa de fraude del 59,78%, muy por encima de todo lo demás (la siguiente categoría, "Music Stores", ya cae a 37,25%, y el resto está en un solo dígito). Con 276 transacciones en esa categoría, no es ruido estadístico — es una señal fuerte y aislada que destaca claramente del resto.

Esto es exactamente el tipo de resultado que un equipo de fraude usaría para priorizar reglas de revisión automática (ej. "toda transacción con MCC de líneas de crucero pasa por revisión manual"), dado que el resto de categorías, aunque tienen algo de fraude, están en un orden de magnitud completamente distinto.

Con esto cerramos el Bloque 5. Tenemos ya 3 tablas nuevas: comparativa_fraude, fraude_por_franja, fraude_por_categoria.

#### Exportación a Excel

##### Perfecto, vamos a consolidar todas las tablas generadas en un único archivo Excel con una hoja por tabla. Antes del código, un par de aclaraciones necesarias:

Qué se exporta y qué no: las tablas de origen (users, cards, transactions) no las exportamos, porque ya cumplieron su función (fueron la base para calcular todo lo demás) y transactions tiene 13,3M de filas — muy por encima del límite de Excel. Solo exportamos las tablas resumen/agregadas que construimos en cada bloque, que es justo el objetivo de todo este trabajo.

Límite de nombres de hoja: Excel no permite nombres de hoja de más de 31 caracteres, así que voy a acortar alguno para evitar errores.

In [47]:
# Diccionario con todas las tablas a exportar, organizadas por bloque
tablas_a_exportar = {
    # Bloque 2 - Comportamiento de gasto
    "Gasto_por_usuario": resumen_gasto_usuario_completo,
    "Gasto_usuario_categoria": gasto_usuario_categoria,
    "Gasto_por_segmento_ingreso": patron_gasto_por_segmento,

    # Bloque 3 - Análisis por tarjeta
    "Resumen_por_tarjeta": resumen_gasto_tarjeta,
    "Comparativa_tipo_tarjeta": comparativa_tipo_tarjeta_v2,
    "Antiguedad_vs_actividad": antiguedad_vs_actividad,

    # Bloque 4 - Análisis temporal
    "Evolucion_mensual": evolucion_mensual,
    "Estacionalidad": estacionalidad,

    # Bloque 5 - Fraude
    "Comparativa_fraude": comparativa_fraude,
    "Fraude_por_franja_horaria": fraude_por_franja,
    "Fraude_por_categoria": fraude_por_categoria,
}

# Verificamos que ningún nombre de hoja supere el límite de 31 caracteres de Excel
for nombre in tablas_a_exportar:
    if len(nombre) > 31:
        print(f"⚠️ Nombre demasiado largo ({len(nombre)} caracteres): {nombre}")
    else:
        print(f"✅ {nombre} ({len(nombre)} caracteres)")

✅ Gasto_por_usuario (17 caracteres)
✅ Gasto_usuario_categoria (23 caracteres)
✅ Gasto_por_segmento_ingreso (26 caracteres)
✅ Resumen_por_tarjeta (19 caracteres)
✅ Comparativa_tipo_tarjeta (24 caracteres)
✅ Antiguedad_vs_actividad (23 caracteres)
✅ Evolucion_mensual (17 caracteres)
✅ Estacionalidad (14 caracteres)
✅ Comparativa_fraude (18 caracteres)
✅ Fraude_por_franja_horaria (25 caracteres)
✅ Fraude_por_categoria (20 caracteres)


##### Código de exportación a Excel

##### Un par de notas sobre el código:

engine="openpyxl": lo especifico explícitamente porque es el motor estándar para escribir .xlsx con pandas; si no lo tienes instalado, avísame y te doy el comando de instalación (pip install openpyxl).


index=False en cada hoja: evita que se exporte la columna de índice automático de pandas (0, 1, 2...), que no aporta nada útil en Excel.
El archivo se guardará en la misma carpeta ruta_datos donde tienes los CSV originales, con el nombre analisis_transacciones_financieras.xlsx.

In [48]:
# Diccionario con todas las tablas a exportar, organizadas por bloque
tablas_a_exportar = {
    # Bloque 2 - Comportamiento de gasto
    "Gasto_por_usuario": resumen_gasto_usuario_completo,
    "Gasto_usuario_categoria": gasto_usuario_categoria,
    "Gasto_por_segmento_ingreso": patron_gasto_por_segmento,

    # Bloque 3 - Análisis por tarjeta
    "Resumen_por_tarjeta": resumen_gasto_tarjeta,
    "Comparativa_tipo_tarjeta": comparativa_tipo_tarjeta_v2,
    "Antiguedad_vs_actividad": antiguedad_vs_actividad,

    # Bloque 4 - Análisis temporal
    "Evolucion_mensual": evolucion_mensual,
    "Estacionalidad": estacionalidad,

    # Bloque 5 - Fraude
    "Comparativa_fraude": comparativa_fraude,
    "Fraude_por_franja_horaria": fraude_por_franja,
    "Fraude_por_categoria": fraude_por_categoria,
}

# Verificación previa de nombres de hoja (por si algún nombre supera 31 caracteres)
for nombre in tablas_a_exportar:
    estado = "⚠️ demasiado largo" if len(nombre) > 31 else "✅ ok"
    print(f"{estado} - {nombre} ({len(nombre)} caracteres)")

# Exportación real: un archivo Excel con una hoja por tabla
ruta_salida = os.path.join(ruta_datos, "analisis_transacciones_financieras.xlsx")

with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    for nombre_hoja, tabla in tablas_a_exportar.items():
        tabla.to_excel(writer, sheet_name=nombre_hoja, index=False)

print(f"\n✅ Archivo exportado correctamente en: {ruta_salida}")

✅ ok - Gasto_por_usuario (17 caracteres)
✅ ok - Gasto_usuario_categoria (23 caracteres)
✅ ok - Gasto_por_segmento_ingreso (26 caracteres)
✅ ok - Resumen_por_tarjeta (19 caracteres)
✅ ok - Comparativa_tipo_tarjeta (24 caracteres)
✅ ok - Antiguedad_vs_actividad (23 caracteres)
✅ ok - Evolucion_mensual (17 caracteres)
✅ ok - Estacionalidad (14 caracteres)
✅ ok - Comparativa_fraude (18 caracteres)
✅ ok - Fraude_por_franja_horaria (25 caracteres)
✅ ok - Fraude_por_categoria (20 caracteres)

✅ Archivo exportado correctamente en: ./data/analisis_transacciones_financieras.xlsx
